<a href="https://colab.research.google.com/github/Girawale-Sanket/Gen-AI-LLM-Assignment-/blob/main/Fine_Tune_Sentiment_Classifier_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate -U
print("Setup Complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.0 MB/s eta 0:00:00
Setup Complete.


In [2]:
train_texts = [
    "This movie was absolutely wonderful, I loved every minute of it.",
    "What a fantastic performance, the whole cast was brilliant.",
    "I really enjoyed this book, could not put it down.",
    "The food at this restaurant was delicious and beautifully presented.",
    "Great service, friendly staff, and a lovely atmosphere overall.",
    "This phone exceeded my expectations in every possible way.",
    "An outstanding film with a heartwarming and memorable story.",
    "The hotel room was clean, spacious, and had a stunning view.",
    "I am thrilled with how well this laptop performs every day.",
    "The concert was incredible, the band sounded better than ever.",
    "Such a delightful little cafe, I will definitely be coming back.",
    "The customer support team resolved my issue quickly and politely.",
    "This course explained everything clearly and I learned so much.",
    "The new update makes the app so much faster and easier to use.",
    "Absolutely loved the ending, it tied the whole story together perfectly.",
    "The staff went above and beyond to make our stay comfortable.",
    "A brilliant, thought-provoking novel that I would recommend to anyone.",
    "The pizza was fresh, hot, and full of flavour.",
    "I am very happy with my purchase, it works perfectly.",
    "The trip was wonderful from start to finish.",
    "This movie was boring and the plot made no sense at all.",
    "Terrible customer service, nobody responded to my complaints.",
    "The food arrived cold and tasted quite bland.",
    "I regret buying this phone, it is slow and the battery is weak.",
    "The hotel room was dirty and smelled unpleasant.",
    "What a disappointing performance, the actors seemed uninterested.",
    "The book was confusing and I gave up halfway through.",
    "This app keeps crashing and it is extremely frustrating to use.",
    "The concert was way too loud and poorly organized.",
    "Such an unpleasant experience, the staff were rude to us.",
    "The laptop overheats constantly and the fan is very noisy.",
    "I would not recommend this restaurant, the service was awful.",
    "The ending felt rushed and left too many questions unanswered.",
    "This course was disorganized and hard to follow.",
    "The new update broke several features that used to work fine.",
    "The room was freezing and the heater did not work at all.",
    "I am very unhappy with this purchase, it broke within a week.",
    "The trip was stressful and nothing went according to plan.",
    "The pizza was soggy and clearly not freshly made.",
    "Nobody at the store seemed willing to help us at all.",
    "I loved python programming",
    "I like punjabi songs",
    "The grapes are sour",
    "Cricket is popular game in India"
]

train_labels = [1]*22 + [0]*22

test_texts = [
    "I had a great time at the beach with my family.",
    "The new restaurant downtown serves amazing pasta.",
    "This gadget works exactly as advertised, very pleased.",
    "The lecture was engaging and easy to understand.",
    "Everything about this vacation exceeded my expectations.",
    "The product stopped working after just two days.",
    "I found the customer service painfully slow and unhelpful.",
    "The movie dragged on and felt far too long.",
    "This laptop keeps freezing and it is very annoying.",
    "The meal was overpriced and honestly quite tasteless.",
    "I like punjabi songs",
    "Cricket is popular game in India"
]
test_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0,1,1]

print(f"Training examples: {len(train_texts)}  |  Test examples: {len(test_texts)}")

Training examples: 44  |  Test examples: 12


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_ds = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenizer_fn(batch):
  return tokenizer(batch["text"], truncation = True, padding="max_length", max_length=12)

train_ds = train_ds.map(tokenizer_fn, batched=True)
test_ds = test_ds.map(tokenizer_fn, batched=True)

print(train_ds[0].keys())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [4]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Model Loaded")
print("that is  the brand new classification head,which knows nothing yet ")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded
that is  the brand new classification head,which knows nothing yet 


In [5]:
from transformers import TrainingArguments, Trainer

Training_args = TrainingArguments(
    output_dir="./sentiment_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer=Trainer(
    model=model,
    args=Training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,0.690347
10,0.655335
15,0.557566
20,0.471830


TrainOutput(global_step=24, training_loss=0.5573637187480927, metrics={'train_runtime': 18.2413, 'train_samples_per_second': 9.648, 'train_steps_per_second': 1.316, 'total_flos': 546428019456.0, 'train_loss': 0.5573637187480927, 'epoch': 4.0})

In [6]:
import numpy as np

predictions = trainer.predict(test_ds)
predicted_labels = np.argmax(predictions.predictions, axis = -1)
accuracy = (predicted_labels == np.array(test_labels)).mean()

print(f"Test Accuracy: {accuracy: .0%}\n")

for text, true_label, pred_label in zip(test_texts, test_labels, predicted_labels):
  mark = "✅" if true_label == pred_label else "❌"
  print(f"{mark} | true: {true_label} | Pred: {pred_label}  {text}")

Test Accuracy:  83%

✅ | true: 1 | Pred: 1  I had a great time at the beach with my family.
✅ | true: 1 | Pred: 1  The new restaurant downtown serves amazing pasta.
✅ | true: 1 | Pred: 1  This gadget works exactly as advertised, very pleased.
✅ | true: 1 | Pred: 1  The lecture was engaging and easy to understand.
✅ | true: 1 | Pred: 1  Everything about this vacation exceeded my expectations.
✅ | true: 0 | Pred: 0  The product stopped working after just two days.
✅ | true: 0 | Pred: 0  I found the customer service painfully slow and unhelpful.
✅ | true: 0 | Pred: 0  The movie dragged on and felt far too long.
✅ | true: 0 | Pred: 0  This laptop keeps freezing and it is very annoying.
✅ | true: 0 | Pred: 0  The meal was overpriced and honestly quite tasteless.
❌ | true: 1 | Pred: 0  I like punjabi songs
❌ | true: 1 | Pred: 0  Cricket is popular game in India


In [7]:
trainer.save_model("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")
print("Saved to ./sentiment_model - this folder now contained YOUR fine.tuned model.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./sentiment_model - this folder now contained YOUR fine.tuned model.
